# Step 6: The first ANN (MLP)
A neural network 802 -> 256 -> 128 -> 1 on the averaged features, compared with XGBoost on the same random split (development stage).
*Same results as `src/06_deep_learning.py`.*

**In short** (what this notebook shows)

- **What:** The first neural network (ANN / MLP) compared with XGBoost.
- **Needs the NoXi data?** Yes. Random split (development step).
- **Result / use:** ANN 0.849.

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: **24 conversations** (English 026, 027, 028, 029, 030, 058, 062, 063; French 001, 007, 009, 011, 013, 014, 020, 021; German 068, 070, 072, 073, 074, 075, 076, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

**Imports** and a fixed random seed (same weights every run).

In [3]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
keras.utils.set_random_seed(42)

In [4]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Load the data** (802 voice and face features) and make a **random 80/20 split**. The scaler learns only from the training part.

In [5]:
df = pd.read_parquet("data/features/noxi_engagement.parquet")
cols = []
for c in df.columns:
    if c.startswith("voice_") or c.startswith("face_"):
        cols.append(c)
X = df[cols].fillna(0).to_numpy("float32")
y = df["engagement"].to_numpy("float32")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

**Baseline to beat:** XGBoost on exactly the same data.

In [6]:
xgb = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                   colsample_bytree=0.8, tree_method="hist", random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

**The ANN:** 802 -> 256 -> 128 -> 1, ReLU, dropout 0.3, Adam, mean squared error.

In [7]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 256)                 │         205,568 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 238,593 (932.00 KB)

 Trainable params: 238,593 (932.00 KB)

 Non-trainable params: 0 (0.00 B)

**Train 80 epochs** in 4 stages of 20, printing the test CCC after each stage.

In [8]:
for stage in range(4):
    model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=0)
    p = model.predict(X_test, verbose=0).flatten()
    print(f"  epoch {(stage + 1) * 20:3d}  test CCC = {ccc(y_test, p):.3f}")

  epoch  20  test CCC = 0.794


  epoch  40  test CCC = 0.840


  epoch  60  test CCC = 0.858


  epoch  80  test CCC = 0.849


**Both models on the same test set.**

In [9]:
mlp_pred = model.predict(X_test, verbose=0).flatten()
print("model                CCC  Pearson    MAE")
for name, pred in [("XGBoost", xgb_pred), ("MLP (Deep)", mlp_pred)]:
    pearson = np.corrcoef(y_test, pred)[0, 1]
    print(f"{name:16s} {ccc(y_test, pred):7.3f} {pearson:8.3f} {mean_absolute_error(y_test, pred):7.3f}")

model                CCC  Pearson    MAE
XGBoost            0.888    0.897   0.047
MLP (Deep)         0.849    0.872   0.054
